In [1]:
import pandas as pd
import os

os.makedirs("filing_data", exist_ok=True)

url = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv"

sp500 = pd.read_csv(url)

sp500["Symbol"] = (
    sp500["Symbol"]
    .str.replace(".", "-", regex=False)
    .str.upper()
)

sp500.to_csv("filing_data/sp500.csv", index=False)

In [2]:
import requests
import json

HEADERS = {
    "User-Agent": "Arnav Singh arnav2003.singh@gmail.com"
}

url = "https://www.sec.gov/files/company_tickers.json"

r = requests.get(url, headers=HEADERS)
r.raise_for_status()

with open("filing_data/company_tickers.json", "w") as f:
    json.dump(r.json(), f)

In [3]:
with open("filing_data/company_tickers.json") as f:
    companies = json.load(f)

ticker_to_cik = {
    company["ticker"].replace(".", "-").upper():
    str(company["cik_str"]).zfill(10)
    for company in companies.values()
}

In [4]:
def get_latest_10k(cik):

    url = f"https://data.sec.gov/submissions/CIK{cik}.json"

    r = requests.get(url, headers=HEADERS)

    if r.status_code != 200:
        return None

    submission = r.json()

    recent = submission["filings"]["recent"]

    for form, acc, doc, date in zip(
        recent["form"],
        recent["accessionNumber"],
        recent["primaryDocument"],
        recent["filingDate"]
    ):

        if form == "10-K":

            return {
                "date": date,
                "accession": acc,
                "document": doc
            }

    return None

In [5]:
def build_url(cik, filing):

    accession = filing["accession"].replace("-", "")

    cik = str(int(cik))

    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession}/{filing['document']}"
    )

In [6]:
def download_html(url, filename):

    r = requests.get(url, headers=HEADERS)

    if r.status_code != 200:
        return False

    with open(filename, "wb") as f:
        f.write(r.content)

    return True

In [7]:
import time
from tqdm import tqdm
import os

os.makedirs("filings", exist_ok=True)

failed = []

for ticker in tqdm(sp500["Symbol"]):

    ticker = ticker.upper()

    cik = ticker_to_cik.get(ticker)

    if cik is None:
        failed.append((ticker, "No CIK"))
        continue

    filing = get_latest_10k(cik)

    if filing is None:
        failed.append((ticker, "No 10-K"))
        continue

    url = build_url(cik, filing)

    filename = f"filings/{ticker}_{filing['date']}.html"

    success = download_html(url, filename)

    if not success:
        failed.append((ticker, "Download failed"))

    time.sleep(0.2)

100%|██████████| 503/503 [05:39<00:00,  1.48it/s]
